In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

ROOT = Path("/Users/alexgonzalez/Documents/nba_quant")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.assists import (
    CURRENT_ASSISTS_FEATURES,
    add_assists_features,
)
from src.features.minutes.rolling import prior_ewm, prior_expanding, prior_shift


In [ ]:
s20 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2019-20/regular_season/player_gamelogs.parquet')
s21 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2020-21/regular_season/player_gamelogs.parquet')
s22 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2021-22/regular_season/player_gamelogs.parquet')
s23 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2022-23/regular_season/player_gamelogs.parquet')
s24 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2023-24/regular_season/player_gamelogs.parquet')
s25 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2024-25/regular_season/player_gamelogs.parquet')
s26 = pd.read_parquet('/Users/alexgonzalez/Documents/nba_quant/data/silver/nba/2025-26/regular_season/player_gamelogs.parquet')
df = pd.concat([s20, s21, s22, s23, s24, s25, s26]).reset_index(drop=True)
df['starting'] = (
    df['start_position'].astype('string').str.strip().isin(['G', 'F', 'C']).astype(int)
)

print(f"Number of rows: {len(df)}")
print(f"Number of columns: {len(df.columns)}")
df.sample(5)


In [ ]:
print(f"Total number of duplicates: {df.duplicated(subset=['game_id', 'player_id']).sum()}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df = add_assists_features(df)

df["ast"] = pd.to_numeric(df["ast"], errors="coerce")
ordered = df.sort_values(["player_id", "game_date", "game_id"])
df["ast_lag_1"] = prior_shift(ordered, "ast", ["player_id"])
df["ast_ewm_hl_3"] = prior_ewm(ordered, "ast", ["player_id"], halflife=3)
df["season_ast_mean"] = prior_expanding(
    ordered, "ast", ["player_id", "season_year"], "mean"
)


In [ ]:
# Production assists contract. predicted_minutes_oof stays out until
# chronological out-of-fold minute predictions are joined.
# ast_lag_1, season_ast_mean, and ast_ewm_hl_3 are baselines only.

AST_FEATURES = [
    "starting",
    *[
        column
        for column in CURRENT_ASSISTS_FEATURES
        if column != "predicted_minutes_oof"
    ],
]
print(f"Number of features: {len(AST_FEATURES)}")
AST_FEATURES


In [ ]:
# ── Prop-specific config (NBA AST) ────────────────────────────────────────────

HOLDOUT_SEASON = "2025-26"
ID_COLS = ["game_id", "player_id", "season_year", "player_name", "game_date"]
TARGET_COL = "ast"
ROLE_COL = "starting"

QUANTILES = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

# Starting point is the minutes quantile search. Replace with tune_xgb_quantile
# on the train pool before treating these as an assists-specific fit.
XGB_PARAMS = dict(
    objective="reg:quantileerror",
    n_estimators=1695,
    max_depth=3,
    learning_rate=0.052852647946832816,
    subsample=0.5916232263322159,
    colsample_bytree=0.6284413523882679,
    reg_alpha=0.022871045941195746,
    reg_lambda=2.0960269065270034,
    min_child_weight=8,
    n_jobs=-1,
    random_state=42,
    early_stopping_rounds=50,
)

AST_TIERS = {
    "<2 ast": lambda a: a < 2,
    "2-5 ast": lambda a: (a >= 2) & (a < 5),
    "5-8 ast": lambda a: (a >= 5) & (a < 8),
    "8+ ast": lambda a: a >= 8,
}

ARTIFACT_STEM = "ast_nba_model"

# ── Shared train/eval helpers ─────────────────────────────────────────────────
from models.shared.splits import prepare_splits
from models.shared.train import run_timeseries_cv, run_walk_forward, evaluate_holdout
from models.shared.analysis import run_feature_ablation, analyze_correlations
from models.shared.artifacts import save_model_bundle, load_model_bundle, predict_quantiles
from models.shared.metrics import interval_coverage, pinball_50, pinball_loss


In [ ]:
df = df[(df['minutes'] > 0)]
print(f"After filtering appearances: {df[AST_FEATURES].shape[0]} rows")


In [ ]:
splits = prepare_splits(
    df,
    holdout_season=HOLDOUT_SEASON,
    features=AST_FEATURES,
    target_col=TARGET_COL,
    id_cols=ID_COLS,
    role_col=ROLE_COL,
    extra_keep=["ast", "ast_lag_1", "season_ast_mean", "ast_ewm_hl_3"],
)
ppm_df = splits["ppm_df"]
ppm_holdout = splits["ppm_holdout"]
X = splits["X"]
y = splits["y"]


In [ ]:
# import importlib
# import models.shared.train as train

# importlib.reload(train)
# tune_xgb_quantile = train.tune_xgb_quantile

# result = tune_xgb_quantile(X, y, n_trials=100, n_splits=5)
# XGB_PARAMS = result["best_params"]


In [ ]:
tscv_results = run_timeseries_cv(
    X, y, ppm_df,
    xgb_params=XGB_PARAMS,
    role_col=ROLE_COL,
    tiers=AST_TIERS,
    quantiles=QUANTILES,
)
wf = run_walk_forward(
    X, y, ppm_df,
    xgb_params=XGB_PARAMS,
    role_col=ROLE_COL,
    tiers=AST_TIERS,
    quantiles=QUANTILES,
)

wf_results = wf["wf_results"]
models_last = wf["models_last"]
preds_last = wf["preds_last"]
X_val_last = wf["X_val_last"]
y_val_last = wf["y_val_last"]
starting_last = wf["starting_last"]
last_fold = wf["last_fold"]


### Evaluation

XGBoost quantile models only. Each split is scored with pinball loss at every trained quantile and with interval coverage for the 80% interval (q0.10–q0.90) and the 90% interval (q0.05–q0.95). The holdout also reports empirical calibration at each quantile, P(y ≤ q̂), interval sharpness as the mean and median width in assists, the share of rows where a lower quantile exceeds a higher one, and assist tiers bucketed by predicted q50 rather than actual assists.


In [ ]:
def score_predictions(y_true, preds, split: str):
    """Pinball loss per quantile and central interval coverage."""
    y_true = np.asarray(y_true, dtype=float)
    pinball_rows = []
    for key, pred in preds.items():
        alpha = float(str(key).split("_", 1)[1])
        pinball_rows.append({
            "split": split,
            "quantile": alpha,
            "pinball": pinball_loss(y_true, pred, alpha),
        })
    coverage_rows = []
    for low, high, nominal in ((0.10, 0.90, 0.80), (0.05, 0.95, 0.90)):
        low_key, high_key = f"q_{low:.2f}", f"q_{high:.2f}"
        if low_key not in preds or high_key not in preds:
            continue
        coverage_rows.append({
            "split": split,
            "interval": f"{nominal:.0%}",
            "coverage": interval_coverage(y_true, preds[low_key], preds[high_key]),
            "target": nominal,
        })
    return pd.DataFrame(pinball_rows), pd.DataFrame(coverage_rows)


wf_pinball, wf_coverage = score_predictions(
    y_val_last, preds_last, last_fold["fold"]
)
display(wf_pinball.round(4))
display(wf_coverage.round(4))


In [ ]:
import shap

for q_key, model in models_last.items():
    explainer = shap.TreeExplainer(model, model_output="raw")
    shap_values = explainer(X_val_last)
    print(f"\n── {q_key} ──")
    shap.plots.beeswarm(shap_values, max_display=30, show=False)
    plt.title(f"SHAP beeswarm — {q_key}")
    plt.tight_layout()
    plt.show()


In [ ]:
model = models_last["q_0.50"]
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_val_last)

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
feat_imp = pd.Series(mean_abs_shap, index=AST_FEATURES).sort_values(ascending=True)

ax = feat_imp.plot(kind="barh", figsize=(8, 6))
ax.set_title("Mean |SHAP| — q_0.50")
ax.set_xlabel("Mean absolute SHAP value")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer
from scipy import stats as sp_stats

result = permutation_importance(
    models_last["q_0.50"],
    X_val_last,
    y_val_last,
    n_repeats=30,
    random_state=42,
    scoring=make_scorer(pinball_50),
)

pi_df = (
    pd.DataFrame({
        "feature": X_val_last.columns,
        "importance": result.importances_mean,
        "std": result.importances_std,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
pi_df.index += 1
pi_df["importance"] = pi_df["importance"].round(5)
pi_df["std"] = pi_df["std"].round(5)
pi_df["p_value"] = sp_stats.ttest_1samp(result.importances, 0.0, axis=1).pvalue.round(4)
pi_df["significant"] = pi_df["p_value"] < 0.05

print("Permutation importance (val set) — higher = more important, significant = p<0.05")
print(pi_df.to_string())


In [ ]:
train_mask_last = last_fold["train_mask"]
val_mask_last = last_fold["val_mask"]
X_tr_last = X[train_mask_last]
X_va_last = X[val_mask_last]
y_tr_last = y[train_mask_last]
y_va_last = y[val_mask_last]

ablation_df = run_feature_ablation(
    AST_FEATURES, X_tr_last, y_tr_last, X_va_last, y_va_last,
    xgb_params=XGB_PARAMS,
)


In [ ]:
from scipy import stats

model = models_last["q_0.50"]
_shap_expl = shap.TreeExplainer(model, model_output="raw")
_shap_vals = _shap_expl(X_val_last)
_mean_abs_shap = np.abs(_shap_vals.values).mean(axis=0)

shap_table = pd.DataFrame({
    "feature": X_val_last.columns,
    "mean_abs_shap": _mean_abs_shap,
}).assign(shap_rank=lambda d: d["mean_abs_shap"].rank(method="min", ascending=False).astype(int))

perm_table = (
    pi_df.sort_values("importance", ascending=False)
    .reset_index(drop=True)
    .rename(columns={"importance": "perm_importance", "std": "perm_importance_std"})
)
perm_table["perm_rank"] = np.arange(1, len(perm_table) + 1)
perm_table = perm_table[["feature", "perm_rank", "perm_importance", "perm_importance_std"]]

perm_pv = pd.DataFrame({
    "feature": X_val_last.columns,
    "perm_pvalue": stats.ttest_1samp(result.importances, 0.0, axis=1, nan_policy="omit").pvalue,
})

abla = ablation_df[["feature", "delta_pinball"]].rename(columns={"delta_pinball": "ablation_delta"})

feature_audit = (
    pd.DataFrame({"feature": AST_FEATURES})
    .merge(shap_table, on="feature", how="left")
    .merge(perm_table, on="feature", how="left")
    .merge(perm_pv, on="feature", how="left")
    .merge(abla, on="feature", how="left")
)

_ms = feature_audit["mean_abs_shap"].astype(float)
_mp = feature_audit["perm_importance"].astype(float)
_max_shap = _ms.fillna(0).max()
_max_perm = _mp.fillna(0).max()
feature_audit["shap_norm"] = np.where(_max_shap > 0, _ms.fillna(0) / _max_shap, 0.0)
feature_audit["perm_norm"] = np.where(_max_perm > 0, _mp.fillna(0) / _max_perm, 0.0)
feature_audit["consensus"] = (feature_audit["shap_norm"] + feature_audit["perm_norm"]) / 2
feature_audit = feature_audit.sort_values("consensus", ascending=False).reset_index(drop=True)
feature_audit.index += 1
feature_audit["keep"] = (feature_audit["perm_pvalue"] < 0.05) & (feature_audit["ablation_delta"] > 0)

print(feature_audit.to_string())


In [ ]:
correlated_features = analyze_correlations(
    df, AST_FEATURES, title="Feature Correlation Matrix — NBA AST"
)
correlated_features


In [ ]:
holdout = evaluate_holdout(
    ppm_df,
    ppm_holdout,
    features=AST_FEATURES,
    target_col=TARGET_COL,
    xgb_params=XGB_PARAMS,
    role_col=ROLE_COL,
    tiers=AST_TIERS,
    wf_results=wf_results,
    quantiles=QUANTILES,
    fold_label=f"{HOLDOUT_SEASON} Blind Holdout",
)
models_ho = holdout["models_ho"]
preds_ho = holdout["preds_ho"]
ho_metrics = holdout["ho_metrics"]
X_ho = holdout["X_ho"]
y_ho = holdout["y_ho"]

ho_pinball, ho_coverage = score_predictions(
    y_ho, preds_ho, f"{HOLDOUT_SEASON} holdout"
)
display(ho_pinball.round(4))
display(ho_coverage.round(4))


In [ ]:
y_true = np.asarray(y_ho, dtype=float)
calibration_rows = []
for key, pred in preds_ho.items():
    alpha = float(str(key).split("_", 1)[1])
    pred = np.asarray(pred, dtype=float)
    empirical = float(np.mean(y_true <= pred))
    calibration_rows.append({
        "quantile": alpha,
        "target": alpha,
        "empirical": empirical,
        "delta": empirical - alpha,
    })
calibration = (
    pd.DataFrame(calibration_rows)
    .sort_values("quantile")
    .reset_index(drop=True)
)

sharpness_rows = []
for low, high, nominal in ((0.10, 0.90, 0.80), (0.05, 0.95, 0.90)):
    low_key, high_key = f"q_{low:.2f}", f"q_{high:.2f}"
    if low_key not in preds_ho or high_key not in preds_ho:
        continue
    width = (
        np.asarray(preds_ho[high_key], dtype=float)
        - np.asarray(preds_ho[low_key], dtype=float)
    )
    sharpness_rows.append({
        "interval": f"{nominal:.0%}",
        "coverage": interval_coverage(y_true, preds_ho[low_key], preds_ho[high_key]),
        "target": nominal,
        "mean_width": float(np.mean(width)),
        "median_width": float(np.median(width)),
    })
sharpness = pd.DataFrame(sharpness_rows)

print(f"{HOLDOUT_SEASON} holdout — empirical quantile calibration")
display(calibration.round(4))
print(f"{HOLDOUT_SEASON} holdout — interval sharpness (assists)")
display(sharpness.round(4))


In [ ]:
quantile_keys = sorted(
    preds_ho,
    key=lambda key: float(str(key).split("_", 1)[1]),
)
grid = np.column_stack([
    np.asarray(preds_ho[key], dtype=float) for key in quantile_keys
])
crossed = np.any(np.diff(grid, axis=1) < 0, axis=1)
n_rows = int(len(crossed))
n_crossed = int(crossed.sum())
print(
    f"{HOLDOUT_SEASON} holdout quantile crossing: "
    f"{n_crossed:,} / {n_rows:,} rows ({n_crossed / n_rows:.1%})"
)

pair_rows = []
for left, right in zip(quantile_keys, quantile_keys[1:]):
    pair_rate = float(np.mean(
        np.asarray(preds_ho[left], dtype=float)
        > np.asarray(preds_ho[right], dtype=float)
    ))
    pair_rows.append({
        "lower": left,
        "upper": right,
        "crossing_rate": pair_rate,
    })
crossing_pairs = pd.DataFrame(pair_rows)
display(crossing_pairs.round(4))


In [ ]:
predicted = np.asarray(preds_ho["q_0.50"], dtype=float)
y_true = np.asarray(y_ho, dtype=float)
low_80 = np.asarray(preds_ho["q_0.10"], dtype=float)
high_80 = np.asarray(preds_ho["q_0.90"], dtype=float)
low_90 = np.asarray(preds_ho["q_0.05"], dtype=float)
high_90 = np.asarray(preds_ho["q_0.95"], dtype=float)
width_80 = high_80 - low_80
width_90 = high_90 - low_90

tier_rows = []
for name, fn in AST_TIERS.items():
    mask = np.asarray(fn(predicted), dtype=bool)
    n = int(mask.sum())
    if n == 0:
        continue
    tier_rows.append({
        "tier": name,
        "n": n,
        "share": n / len(predicted),
        "pinball_q50": pinball_loss(y_true[mask], predicted[mask], 0.50),
        "empirical_q50": float(np.mean(y_true[mask] <= predicted[mask])),
        "coverage_80": interval_coverage(y_true[mask], low_80[mask], high_80[mask]),
        "coverage_90": interval_coverage(y_true[mask], low_90[mask], high_90[mask]),
        "mean_width_80": float(np.mean(width_80[mask])),
        "mean_width_90": float(np.mean(width_90[mask])),
    })
tier_report = pd.DataFrame(tier_rows)
print(f"{HOLDOUT_SEASON} holdout — assist tiers by predicted q50")
display(tier_report.round(4))


In [ ]:
N_BOOT = 2_000
SEED = 42

y = np.asarray(y_ho, dtype=float)
predictors = {
    "model": np.asarray(preds_ho["q_0.50"], dtype=float),
    "season_average": ppm_holdout["season_ast_mean"].to_numpy(dtype=float),
    "last_game": ppm_holdout["ast_lag_1"].to_numpy(dtype=float),
    "ewma": ppm_holdout["ast_ewm_hl_3"].to_numpy(dtype=float),
}
dates = ppm_holdout["game_date"].to_numpy()
finite = np.isfinite(y)
for pred in predictors.values():
    finite &= np.isfinite(pred)
n_dropped = int((~finite).sum())

y = y[finite]
dates = dates[finite]
predictors = {name: pred[finite] for name, pred in predictors.items()}
codes, _unique_dates = pd.factorize(dates, sort=False)
n_clusters = int(codes.max()) + 1
abs_err = {name: np.abs(y - pred) for name, pred in predictors.items()}
cluster_n = np.bincount(codes).astype(float)
cluster_sum = {
    name: np.bincount(codes, weights=err) for name, err in abs_err.items()
}

rng = np.random.default_rng(SEED)
draws = rng.integers(0, n_clusters, size=(N_BOOT, n_clusters))
weights = np.zeros((N_BOOT, n_clusters), dtype=np.int32)
np.add.at(
    weights,
    (np.repeat(np.arange(N_BOOT), n_clusters), draws.ravel()),
    1,
)
denom = weights @ cluster_n
boot_mae = {
    name: (weights @ cluster_sum[name]) / denom for name in predictors
}
point = {name: float(err.mean()) for name, err in abs_err.items()}

labels = {
    "model": "Model (q50)",
    "season_average": "Season average",
    "last_game": "Last-game assists",
    "ewma": "EWMA (halflife 3)",
}
order = ["model", "season_average", "last_game", "ewma"]
mae_table = pd.DataFrame([
    {
        "predictor": labels[name],
        "mae": point[name],
        "ci_low": float(np.quantile(boot_mae[name], 0.025)),
        "ci_high": float(np.quantile(boot_mae[name], 0.975)),
    }
    for name in order
])
delta_table = pd.DataFrame([
    {
        "comparison": f"model − {labels[name]}",
        "delta_mae": point["model"] - point[name],
        "ci_low": float(np.quantile(boot_mae["model"] - boot_mae[name], 0.025)),
        "ci_high": float(np.quantile(boot_mae["model"] - boot_mae[name], 0.975)),
    }
    for name in order[1:]
])

print(
    f"{HOLDOUT_SEASON} holdout MAE — paired bootstrap clustered by game date\n"
    f"  {N_BOOT:,} resamples | {n_clusters:,} dates | {len(y):,} rows"
    f" | dropped {n_dropped:,} rows with a missing baseline"
)
display(mae_table.round(3))
print("Paired difference (negative means the model has lower MAE)")
display(delta_table.round(3))


### Assists acceptance report

Identical holdout rows for the quantile model against last-game assists, the season-to-date average, and EWMA. Distribution scores, per-quantile calibration, and pregame slices follow. Walk-forward fold MAEs use the saved q50 pinball.


In [ ]:
N_BOOT = 2_000
SEED = 42
WIS_INTERVALS = (
    (0.80, "q_0.40", "q_0.60"),
    (0.60, "q_0.30", "q_0.70"),
    (0.40, "q_0.20", "q_0.80"),
    (0.20, "q_0.10", "q_0.90"),
    (0.10, "q_0.05", "q_0.95"),
)

y_all = np.asarray(y_ho, dtype=float)
quantiles = {key: np.asarray(pred, dtype=float) for key, pred in preds_ho.items()}
alpha_of = {key: float(str(key).split("_", 1)[1]) for key in quantiles}
ordered_keys = sorted(quantiles, key=alpha_of.get)
model = quantiles["q_0.50"]
baselines = {
    "Last-game assists": ppm_holdout["ast_lag_1"].to_numpy(dtype=float),
    "Season-to-date average": ppm_holdout["season_ast_mean"].to_numpy(dtype=float),
    "EWMA assists": ppm_holdout["ast_ewm_hl_3"].to_numpy(dtype=float),
}
paired = np.isfinite(y_all) & np.isfinite(model)
for pred in baselines.values():
    paired &= np.isfinite(pred)

def point_row(name, y, pred):
    error = y - pred
    absolute = np.abs(error)
    return {
        "model": name,
        "mae": float(absolute.mean()),
        "median_ae": float(np.median(absolute)),
        "mean_bias": float(error.mean()),
    }

def interval_score(y, lower, upper, alpha):
    width = upper - lower
    below = y < lower
    above = y > upper
    return (
        width
        + (2.0 / alpha) * (lower - y) * below
        + (2.0 / alpha) * (y - upper) * above
    )

def wis_values(y, qmap):
    present = [
        (alpha, qmap[low], qmap[high])
        for alpha, low, high in WIS_INTERVALS
        if low in qmap and high in qmap
    ]
    total = 0.5 * np.abs(y - qmap["q_0.50"])
    for alpha, lower, upper in present:
        total = total + (alpha / 2.0) * interval_score(y, lower, upper, alpha)
    return total / (len(present) + 0.5)

def clustered_mae_bootstrap(y, predictors, dates, n_boot, seed):
    codes, _ = pd.factorize(dates, sort=False)
    n_clusters = int(codes.max()) + 1
    cluster_n = np.bincount(codes).astype(float)
    cluster_sum = {
        name: np.bincount(codes, weights=np.abs(y - pred))
        for name, pred in predictors.items()
    }
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, n_clusters, size=(n_boot, n_clusters))
    weights = np.zeros((n_boot, n_clusters), dtype=np.int32)
    np.add.at(
        weights,
        (np.repeat(np.arange(n_boot), n_clusters), draws.ravel()),
        1,
    )
    denom = weights @ cluster_n
    return {name: (weights @ total) / denom for name, total in cluster_sum.items()}

y_paired = y_all[paired]
paired_preds = {"Quantile XGBoost q50": model[paired]}
paired_preds.update({name: pred[paired] for name, pred in baselines.items()})
dates_paired = pd.to_datetime(ppm_holdout["game_date"]).to_numpy()[paired]
boot_mae = clustered_mae_bootstrap(
    y_paired, paired_preds, dates_paired, N_BOOT, SEED
)

point_order = [
    "Last-game assists",
    "Season-to-date average",
    "EWMA assists",
    "Quantile XGBoost q50",
]
point_table = pd.DataFrame([
    point_row(name, y_paired, paired_preds[name]) for name in point_order
])
best_name = min(baselines, key=lambda name: point_table.loc[
    point_table["model"] == name, "mae"
].iloc[0])
model_boot = boot_mae["Quantile XGBoost q50"]
delta_table = pd.DataFrame([
    {
        "comparison": f"q50 − {name}",
        "delta_mae": float(
            point_table.loc[point_table["model"] == "Quantile XGBoost q50", "mae"].iloc[0]
            - point_table.loc[point_table["model"] == name, "mae"].iloc[0]
        ),
        "ci_low": float(np.quantile(model_boot - boot_mae[name], 0.025)),
        "ci_high": float(np.quantile(model_boot - boot_mae[name], 0.975)),
    }
    for name in point_order[:-1]
])

fold_rows = []
for fold in wf_results:
    part = ppm_df.loc[fold["val_mask"]]
    y_fold = part[TARGET_COL].to_numpy(dtype=float)
    fold_row = {
        "fold": fold["fold"],
        "n": int(len(part)),
        "model_mae": 2.0 * float(fold["pinball"]),
    }
    for name, column in (
        ("last_game", "ast_lag_1"),
        ("season_average", "season_ast_mean"),
        ("ewma", "ast_ewm_hl_3"),
    ):
        pred = part[column].to_numpy(dtype=float)
        ok = np.isfinite(y_fold) & np.isfinite(pred)
        fold_row[name] = float(np.mean(np.abs(y_fold[ok] - pred[ok])))
    fold_row["best_baseline"] = min(
        fold_row["last_game"], fold_row["season_average"], fold_row["ewma"]
    )
    fold_row["model_minus_best"] = fold_row["model_mae"] - fold_row["best_baseline"]
    fold_rows.append(fold_row)
fold_table = pd.DataFrame(fold_rows)

print("1. Point accuracy versus baselines")
print(
    f"Identical holdout rows: {int(paired.sum()):,}  |  "
    f"dropped {int((~paired).sum()):,} with a missing baseline  |  "
    f"{N_BOOT:,} date-clustered resamples"
)
print(
    "Mean bias is actual minus predicted. "
    f"q50 pinball {pinball_loss(y_paired, paired_preds['Quantile XGBoost q50'], 0.50):.4f} "
    "equals half the q50 MAE."
)
display(point_table.round(3))
print("Paired MAE difference, clustered by game date. Negative means q50 is lower.")
display(delta_table.round(3))
print(
    "Walk-forward model MAE is 2 × q50 pinball on every validation row. "
    "Baseline MAE drops rows with no prior."
)
display(fold_table.round(3))

print("\n2. Distribution quality")
width_80 = quantiles["q_0.90"] - quantiles["q_0.10"]
width_90 = quantiles["q_0.95"] - quantiles["q_0.05"]
grid = np.column_stack([quantiles[key] for key in ordered_keys])
crossed = np.any(np.diff(grid, axis=1) < 0, axis=1)
distribution = pd.DataFrame([
    {
        "cov_80": interval_coverage(y_all, quantiles["q_0.10"], quantiles["q_0.90"]),
        "width_80_mean": float(width_80.mean()),
        "width_80_median": float(np.median(width_80)),
        "cov_90": interval_coverage(y_all, quantiles["q_0.05"], quantiles["q_0.95"]),
        "width_90_mean": float(width_90.mean()),
        "width_90_median": float(np.median(width_90)),
        "wis": float(np.mean(wis_values(y_all, quantiles))),
        "quantile_crossing": float(crossed.mean()),
    }
])
display(distribution.round(4))
print(
    f"Best baseline MAE {point_table.loc[point_table['model'] == best_name, 'mae'].iloc[0]:.3f} "
    f"({best_name}) is that point forecast's CRPS. Model WIS is the distribution score."
)

print("\n3. Calibration of every quantile")
calibration = pd.DataFrame([
    {
        "quantile": f"q{alpha_of[key] * 100:02.0f}",
        "expected": alpha_of[key],
        "observed": float(np.mean(y_all <= quantiles[key])),
        "difference": float(np.mean(y_all <= quantiles[key]) - alpha_of[key]),
    }
    for key in ordered_keys
])
display(calibration.round(4))

def subgroup_row(name, mask):
    mask = np.asarray(mask, dtype=bool) & np.isfinite(y_all) & np.isfinite(model)
    n = int(mask.sum())
    if n == 0:
        return None
    y = y_all[mask]
    qmap = {key: values[mask] for key, values in quantiles.items()}
    error = y - qmap["q_0.50"]
    return {
        "group": name,
        "n": n,
        "mae": float(np.mean(np.abs(error))),
        "bias": float(np.mean(error)),
        "wis": float(np.mean(wis_values(y, qmap))),
        "cov_80": interval_coverage(y, qmap["q_0.10"], qmap["q_0.90"]),
        "width_80_mean": float(np.mean(qmap["q_0.90"] - qmap["q_0.10"])),
        "width_80_median": float(np.median(qmap["q_0.90"] - qmap["q_0.10"])),
        "cov_90": interval_coverage(y, qmap["q_0.05"], qmap["q_0.95"]),
        "width_90_mean": float(np.mean(qmap["q_0.95"] - qmap["q_0.05"])),
        "width_90_median": float(np.median(qmap["q_0.95"] - qmap["q_0.05"])),
    }

print("\n4. Pregame-defined subgroups")
starting = ppm_holdout["starting"].to_numpy()
months = pd.to_datetime(ppm_holdout["game_date"]).dt.to_period("M").astype(str)
subgroup_rows = [
    subgroup_row("Confirmed starter", starting == 1),
    subgroup_row("Bench", starting == 0),
]
for name, fn in AST_TIERS.items():
    subgroup_rows.append(subgroup_row(f"Predicted q50 {name}", fn(model)))
for month in sorted(months.unique()):
    subgroup_rows.append(subgroup_row(str(month), months.to_numpy() == month))
subgroup_table = pd.DataFrame([row for row in subgroup_rows if row is not None])
display(subgroup_table.round(3))

print("\n5. Tail failures")
absolute_error = np.abs(y_all - model)
tail_table = pd.DataFrame([
    {
        "threshold": f">{cutoff:g} ast",
        "rate": float(np.mean(absolute_error > cutoff)),
        "n": int(np.sum(absolute_error > cutoff)),
    }
    for cutoff in (2, 4, 6)
])
display(tail_table.round(4))

q50_rate = float(calibration.loc[calibration["quantile"] == "q50", "observed"].iloc[0])
max_gap = float(calibration["difference"].abs().max())
best_delta = delta_table.loc[delta_table["comparison"] == f"q50 − {best_name}"].iloc[0]
model_wis = float(distribution["wis"].iloc[0])
best_mae = float(point_table.loc[point_table["model"] == best_name, "mae"].iloc[0])
print("\nChecks from this frame")
print(f"  q50 observed rate {q50_rate:.1%}  (target about 48–52%)")
print(f"  largest |quantile gap| {max_gap:.1%}")
print(
    f"  q50 vs {best_name}: delta MAE {best_delta['delta_mae']:+.3f} "
    f"[{best_delta['ci_low']:+.3f}, {best_delta['ci_high']:+.3f}]"
)
print(f"  model WIS {model_wis:.3f} vs best baseline CRPS/MAE {best_mae:.3f}")
print(
    "  walk-forward folds with model MAE below the best baseline: "
    f"{int((fold_table['model_minus_best'] < 0).sum())} / {len(fold_table)}"
)
print(f"  raw quantile crossing {float(crossed.mean()):.2%} of rows")


### Follow-ups before freezing

April versus the rest of the regular season, starter and bench median calibration, monotone quantile rearrangement, and the cold-start rows.


In [ ]:
import importlib

import models.shared.artifacts as artifact_mod

importlib.reload(artifact_mod)
from models.shared.artifacts import monotonize_quantiles

dates = pd.to_datetime(ppm_holdout["game_date"])
y_all = np.asarray(y_ho, dtype=float)
raw_q = {key: np.asarray(pred, dtype=float) for key, pred in preds_ho.items()}
model = raw_q["q_0.50"]

def slice_score(name, mask, qmap):
    mask = np.asarray(mask, dtype=bool)
    n = int(mask.sum())
    if n == 0:
        return {"slice": name, "n": 0}
    y = y_all[mask]
    qmap = {key: values[mask] for key, values in qmap.items()}
    error = y - qmap["q_0.50"]
    return {
        "slice": name,
        "n": n,
        "mae": float(np.mean(np.abs(error))),
        "bias": float(np.mean(error)),
        "p_actual_le_q50": float(np.mean(y <= qmap["q_0.50"])),
        "cov_80": interval_coverage(y, qmap["q_0.10"], qmap["q_0.90"]),
        "mean_width_80": float(np.mean(qmap["q_0.90"] - qmap["q_0.10"])),
        "wis": float(np.mean(wis_values(y, qmap))),
    }

print("1. April is late regular season, not the postseason")
holdout_season = df.loc[df["season_year"].astype("string").eq(str(HOLDOUT_SEASON))]
print(holdout_season["season_type"].astype("string").value_counts(dropna=False).to_string())
print(
    f"Holdout dates {dates.min().date()} → {dates.max().date()}. "
    "The silver store only has regular_season gamelogs, so playoff games are not in this test."
)
blocks = {
    "October–February": dates < "2026-03-01",
    "March 1–14": (dates >= "2026-03-01") & (dates < "2026-03-15"),
    "March 15–31": (dates >= "2026-03-15") & (dates < "2026-04-01"),
    "April regular season": dates >= "2026-04-01",
}
calendar = pd.DataFrame([slice_score(name, mask, raw_q) for name, mask in blocks.items()])
display(calendar.round(3))
print(
    "Limitation: April's higher MAE and lower 80% coverage is the last two weeks of the "
    "regular season. It is not a postseason effect, and the model has not been tested on playoff games."
)

print("\n2. Starter and bench mean bias versus median calibration")
starting = ppm_holdout["starting"].to_numpy()
role = pd.DataFrame([
    slice_score("Confirmed starter", starting == 1, raw_q),
    slice_score("Bench", starting == 0, raw_q),
])
display(role.round(3))
for _, row in role.iterrows():
    if row["n"] == 0:
        continue
    gap = row["p_actual_le_q50"] - 0.50
    if abs(gap) <= 0.02:
        reading = "q50 is near 50%, so the mean bias is skew around a calibrated median"
    elif gap > 0:
        reading = "q50 is too high: actuals fall under the median more often than 50%"
    else:
        reading = "q50 is too low: actuals fall under the median less often than 50%"
    print(
        f"  {row['slice']}: bias {row['bias']:+.2f} ast, "
        f"P(actual <= q50) {row['p_actual_le_q50']:.1%}. {reading}."
    )

print("\n3. Monotone rearrangement")
ordered_keys = sorted(raw_q, key=lambda key: float(str(key).split("_", 1)[1]))
grid = np.column_stack([raw_q[key] for key in ordered_keys])
crossed = np.any(np.diff(grid, axis=1) < 0, axis=1)
preds_ho_mono = monotonize_quantiles(raw_q)
mono_grid = np.column_stack([preds_ho_mono[key] for key in ordered_keys])
shift = pd.DataFrame([
    {
        "quantile": key,
        "mean_abs_shift": float(np.mean(np.abs(preds_ho_mono[key] - raw_q[key]))),
        "share_changed": float(np.mean(preds_ho_mono[key] != raw_q[key])),
    }
    for key in ordered_keys
])
print(
    f"Crossing before {crossed.mean():.2%} of rows; "
    f"after rearrangement {np.any(np.diff(mono_grid, axis=1) < 0, axis=1).mean():.2%}."
)
display(shift.round(4))
print("Simulate from preds_ho_mono. The saved trees stay raw; sort each row after predict.")

print("\n4. Cold-start and missing-baseline rows")
lag = ppm_holdout["ast_lag_1"].to_numpy(dtype=float)
season_avg = ppm_holdout["season_ast_mean"].to_numpy(dtype=float)
ewma = ppm_holdout["ast_ewm_hl_3"].to_numpy(dtype=float)
cold = ~np.isfinite(lag)
season_opener = np.isfinite(lag) & ~np.isfinite(season_avg)
missing_any = cold | season_opener | ~np.isfinite(ewma)
print(
    f"Missing any baseline: {int(missing_any.sum()):,} | "
    f"no prior game (ast_lag_1 missing): {int(cold.sum()):,} | "
    f"season opener with history: {int(season_opener.sum()):,}"
)

role_quantiles = {}
for role_value, role_name in ((1, "starter"), (0, "bench")):
    history = ppm_df.loc[ppm_df["starting"] == role_value, "ast"].to_numpy(dtype=float)
    history = history[np.isfinite(history)]
    role_quantiles[role_name] = {
        key: float(np.quantile(history, float(str(key).split("_", 1)[1])))
        for key in ordered_keys
    }
fallback_q = {key: preds_ho_mono[key].copy() for key in ordered_keys}
for key in ordered_keys:
    fallback_q[key][cold & (starting == 1)] = role_quantiles["starter"][key]
    fallback_q[key][cold & (starting == 0)] = role_quantiles["bench"][key]

missing_table = pd.DataFrame([
    slice_score("Season opener, model", season_opener, raw_q),
    slice_score("No prior game, model", cold, raw_q),
    slice_score("No prior game, role fallback", cold, fallback_q),
    slice_score("Any missing baseline, model", missing_any, raw_q),
])
display(missing_table.round(3))
print(
    "Live fallback: if ast_lag_1 exists, predict with the quantile model and then "
    "sort that row into increasing quantile order, even when season_ast_mean is missing. "
    "If ast_lag_1 is missing, do not use the tree. Use the training-pool assist quantiles "
    "for the confirmed role (starter or bench). preds_ho_live applies that rule."
)
preds_ho_live = fallback_q


### Assists model checkpoint

Do not refit or tune this fit on the opened 2025–26 holdout once the acceptance report is signed off. Two later checks stay open: a probabilistic EWMA baseline, and the players bookmakers actually post. The second one is not in this data. Until a line file exists, the proxy is pregame EWMA assists, not the assists recorded that night.


In [ ]:
LINE_PROXY_CUTOFFS = (3, 6)

train_y = ppm_df["ast"].to_numpy(dtype=float)
train_ewma = ppm_df["ast_ewm_hl_3"].to_numpy(dtype=float)
train_ok = np.isfinite(train_y) & np.isfinite(train_ewma)
train_residual = train_y[train_ok] - train_ewma[train_ok]

hold_y = np.asarray(y_ho, dtype=float)
hold_ewma = ppm_holdout["ast_ewm_hl_3"].to_numpy(dtype=float)
hold_season = ppm_holdout["season_ast_mean"].to_numpy(dtype=float)
model_q = {key: np.asarray(pred, dtype=float) for key, pred in preds_ho.items()}
levels = sorted(model_q, key=lambda key: float(str(key).split("_", 1)[1]))
residual_q = {
    key: float(np.quantile(train_residual, float(str(key).split("_", 1)[1])))
    for key in levels
}
ewma_q = {key: hold_ewma + shift for key, shift in residual_q.items()}
scored = np.isfinite(hold_y) & np.isfinite(hold_ewma)

def _wis(mask, qmap):
    y = hold_y[mask]
    sliced = {key: values[mask] for key, values in qmap.items()}
    return float(np.mean(wis_values(y, sliced)))

model_wis_all = _wis(np.isfinite(hold_y), model_q)
model_wis_paired = _wis(scored, model_q)
ewma_wis_paired = _wis(scored, ewma_q)
baseline = pd.DataFrame([
    {
        "distribution": "Quantile XGBoost",
        "rows": "all holdout",
        "n": int(np.isfinite(hold_y).sum()),
        "wis": model_wis_all,
        "cov_80": interval_coverage(
            hold_y, model_q["q_0.10"], model_q["q_0.90"]
        ),
        "mean_width_80": float(np.mean(model_q["q_0.90"] - model_q["q_0.10"])),
    },
    {
        "distribution": "Quantile XGBoost",
        "rows": "EWMA available",
        "n": int(scored.sum()),
        "wis": model_wis_paired,
        "cov_80": interval_coverage(
            hold_y[scored], model_q["q_0.10"][scored], model_q["q_0.90"][scored]
        ),
        "mean_width_80": float(
            np.mean(model_q["q_0.90"][scored] - model_q["q_0.10"][scored])
        ),
    },
    {
        "distribution": "EWMA + train residuals",
        "rows": "EWMA available",
        "n": int(scored.sum()),
        "wis": ewma_wis_paired,
        "cov_80": interval_coverage(
            hold_y[scored], ewma_q["q_0.10"][scored], ewma_q["q_0.90"][scored]
        ),
        "mean_width_80": float(
            np.mean(ewma_q["q_0.90"][scored] - ewma_q["q_0.10"][scored])
        ),
    },
])
print(
    "EWMA quantile distribution = ast_ewm_hl_3 + residual quantiles from the "
    f"training pool only ({int(train_ok.sum()):,} rows). Holdout outcomes are not in that pool."
)
print(
    f"Model WIS on all holdout rows: {model_wis_all:.3f}. "
    f"On the shared EWMA rows, model WIS {model_wis_paired:.3f} vs "
    f"EWMA distribution WIS {ewma_wis_paired:.3f} "
    f"(delta {model_wis_paired - ewma_wis_paired:+.3f}; negative means the model is better)."
)
display(baseline.round(3))

pregame = np.where(np.isfinite(hold_ewma), hold_ewma, hold_season)
proxy_rows = []
for cutoff in LINE_PROXY_CUTOFFS:
    mask = np.isfinite(hold_y) & np.isfinite(pregame) & (pregame >= cutoff)
    y = hold_y[mask]
    q50 = model_q["q_0.50"][mask]
    proxy_rows.append({
        "proxy": f"pregame EWMA >= {cutoff:g}",
        "n": int(mask.sum()),
        "share": float(mask.mean()),
        "mae": float(np.mean(np.abs(y - q50))),
        "wis": _wis(mask, model_q),
        "cov_80": interval_coverage(y, model_q["q_0.10"][mask], model_q["q_0.90"][mask]),
        "mean_width_80": float(np.mean(model_q["q_0.90"][mask] - model_q["q_0.10"][mask])),
    })
print(
    "No assists-line file is available. Bookmakers usually post players expected "
    "to record a regular number of assists. The table uses pregame EWMA (season average if EWMA "
    "is missing), not assists recorded in that game. It is not the posted-line population."
)
display(pd.DataFrame(proxy_rows).round(3))
print(
    "Do not change features, quantiles, or XGB_PARAMS to improve 2025-26 after this "
    "report is accepted. The open checks are this EWMA distribution comparison and, "
    "later, rows that actually had a posted assists line."
)


In [ ]:
save_path = save_model_bundle(
    models_ho,
    train_df=ppm_df,
    holdout_df=ppm_holdout,
    wf_results=wf_results,
    ho_metrics=ho_metrics,
    features=AST_FEATURES,
    artifact_stem=ARTIFACT_STEM,
)

bundle = load_model_bundle(save_path)
models_loaded = bundle["quantile_models"]
feature_names = bundle["feature_names"]

assert list(X_ho.columns) == list(feature_names)
quantile_keys = sorted(
    models_loaded,
    key=lambda key: float(str(key).split("_", 1)[1]),
)
loaded_preds = pd.DataFrame({
    key: models_loaded[key].predict(X_ho) for key in quantile_keys
})
print("Saved quantiles:", ", ".join(loaded_preds.columns))
print(
    f"Loaded model pinball q50 (holdout): "
    f"{pinball_loss(y_ho, loaded_preds['q_0.50'], 0.50):.3f}"
)
print(
    f"Loaded model 80% coverage (holdout): "
    f"{interval_coverage(y_ho, loaded_preds['q_0.10'], loaded_preds['q_0.90']):.1%}"
)
print(f"Trained on data up to      : {bundle['train_end'].date()}")
print(f"Holdout through            : {bundle['val_end'].date()}")

loaded_preds.head(3)
